# Audio Splicing Detection and Localization Using Acoustic Feature Analysis
**Multimedia Forensics Lab — Mini Project**

This notebook is a forensic analysis tool that flags *suspicious edit boundaries* in an audio recording and reports their approximate timestamps.

**Pipeline:** load audio → frame-level acoustic features → feature-discontinuity score → adaptive threshold & peak picking → localized boundaries → evaluation against known ground truth.

**How to use:** `Runtime → Run all`. Sections 1–5 build the tool, section 6 is a quick sanity check, sections 7–8 build a labelled dataset and evaluate it (this is your *Performance evaluation*), section 9 analyzes your own file, section 10 launches a demo web app.

> Scope: this is a prototype that flags *possible* edits. A flagged boundary is not proof of tampering.

## 1. Setup

In [ ]:
%pip -q install librosa soundfile scipy pandas matplotlib plotly "gradio>=4.40"

In [ ]:
import time, warnings, io
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import librosa, librosa.display
import soundfile as sf
from scipy.signal import find_peaks
from IPython.display import Audio, display
warnings.filterwarnings("ignore")

try:
    from google.colab import files      # only exists inside Colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# ---------------- Analysis settings ----------------
SR = 16000            # speech is band-limited; 16 kHz is the standard for speech forensics
FRAME_LENGTH = 512    # 32 ms analysis frame
HOP_LENGTH = 160      # 10 ms hop  -> 100 frames per second
N_MFCC = 13
RNG = np.random.default_rng(42)

print("librosa", librosa.__version__, "| running in Colab:", IN_COLAB)

## 2. Acoustic feature extraction  *(Member 1)*

For every 10 ms frame we compute:

| Feature | What it captures | Why it matters for splicing |
|---|---|---|
| 13 MFCCs | spectral envelope (vocal tract + **microphone/channel**) | a pasted segment usually comes from a different mic/room |
| log RMS energy | loudness | gain mismatch between recordings |
| spectral centroid | "brightness" | different channel bandwidth / EQ |
| spectral flatness | noise-likeness of the spectrum | different background noise |
| zero-crossing rate | rough frequency content | noise / fricative changes |
| spectral flux | frame-to-frame spectral change | sudden cuts create a burst |
| **noise floor** in 4 bands (0–1, 1–2, 2–4, 4–8 kHz) | background-noise level and colour | **the strongest cue** — background noise rarely matches across recordings |

Total: **22 features per frame**, organised in **4 groups**: MFCC (timbre/channel), level, spectral shape, background noise.

In [ ]:
FEATURE_NAMES = ([f"mfcc_{i}" for i in range(N_MFCC)] +
                 ["log_rms", "centroid_khz", "flatness", "zcr", "flux"] +
                 ["noise_0_1k", "noise_1_2k", "noise_2_4k", "noise_4_8k"])
NOISE_BANDS = [(0, 1000), (1000, 2000), (2000, 4000), (4000, 8000)]
FEATURE_GROUPS = {                       # column indices of each group
    "MFCC (timbre / channel)": list(range(0, 13)),
    "Level": [13],
    "Spectral shape": [14, 15, 16, 17],
    "Background noise": [18, 19, 20, 21],
}

def extract_features(y, sr=SR):
    # Return (features [n_frames x 19], frame_times [n_frames]).
    y = np.asarray(y, dtype=np.float32)
    if y.ndim != 1:
        raise ValueError("Audio must be mono.")
    if len(y) < FRAME_LENGTH * 4:
        raise ValueError("Audio is too short for analysis.")

    S = np.abs(librosa.stft(y, n_fft=FRAME_LENGTH, hop_length=HOP_LENGTH))  # magnitude spectrogram
    P = S ** 2

    mfcc = librosa.feature.mfcc(S=librosa.power_to_db(
        librosa.feature.melspectrogram(S=P, sr=sr, n_mels=40)), n_mfcc=N_MFCC)
    rms = librosa.feature.rms(S=S, frame_length=FRAME_LENGTH)[0]
    log_rms = np.log10(rms + 1e-6)
    centroid = librosa.feature.spectral_centroid(S=S, sr=sr)[0] / 1000.0
    flatness = np.log10(librosa.feature.spectral_flatness(S=S)[0] + 1e-6)
    zcr = librosa.feature.zero_crossing_rate(y, frame_length=FRAME_LENGTH, hop_length=HOP_LENGTH)[0]

    logS = np.log10(S + 1e-6)
    flux = np.concatenate([[0.0], np.mean(np.maximum(np.diff(logS, axis=1), 0), axis=0)])

    # Noise floor per band: the 10th-percentile log-magnitude of the bins in that band.
    # Between harmonics and in pauses this is dominated by background noise / mic hiss.
    freqs = librosa.fft_frequencies(sr=sr, n_fft=FRAME_LENGTH)
    noise_floor = np.column_stack([np.percentile(logS[(freqs >= lo) & (freqs < hi)], 10, axis=0)
                                   for lo, hi in NOISE_BANDS])

    n = min(mfcc.shape[1], len(log_rms), len(centroid), len(flatness), len(zcr), len(flux))
    feats = np.column_stack([mfcc[:, :n].T, log_rms[:n], centroid[:n], flatness[:n],
                             zcr[:n], flux[:n], noise_floor[:n]])
    times = librosa.frames_to_time(np.arange(n), sr=sr, hop_length=HOP_LENGTH)
    return feats.astype(np.float32), times

## 3. Splice detection and localization  *(Member 2)*

**Step 1 — robust standardization.** Each feature becomes a robust z-score (median/MAD) and is clipped to ±4, so no feature dominates and single loud clicks can't swing the result.

**Step 2 — two-window discontinuity score per feature group.** At each time $t$, average the features over $W$ frames to the left and to the right. For each group $g$ (MFCC, level, spectral shape, background noise):

$$D_g(t)=\sqrt{\frac{1}{|g|}\sum_{j\in g}\big(\mu_{j}^{\text{right}}(t)-\mu_{j}^{\text{left}}(t)\big)^2}$$

**Step 3 — combine groups with max.** $D(t)=\max_g D_g(t)$. A splice usually breaks *one* property strongly (often background noise); averaging all 22 features would dilute that signal under phoneme-to-phoneme MFCC variation. We also return which group caused the flag — that's the *explanation* an investigator needs.

*(Design note for your viva: we tried median windows — they give a flat-topped peak ±W/2 wide and localize badly — and averaging all features — it buries the noise cue. Max-over-groups with clipped means worked best on validation.)*

**Step 4 — adaptive threshold.** $\tau = \operatorname{median}(D) + k\cdot 1.4826\cdot\operatorname{MAD}(D)$. The threshold adapts to each recording, so a noisy file doesn't flag everything.

**Step 5 — peak picking + non-maximum suppression.** Only local maxima above $\tau$, at least `min_separation` seconds apart, become *suspicious boundaries*. This stops one edit producing a dozen alerts.

Parameters: `window_seconds` (W = 0.5 s), `threshold_factor` (k), `min_separation_seconds`. We tune **k** on a validation split in section 8 — never on the test set.

In [ ]:
def robust_standardize(F, clip=4.0):
    med = np.median(F, axis=0)
    mad = np.median(np.abs(F - med), axis=0) * 1.4826
    return np.clip((F - med) / (mad + 1e-6), -clip, clip)

def discontinuity_scores(Z, win):
    # Returns D(t) [n] and the per-group scores [n x groups]. Edge frames get 0.
    n = len(Z)
    csum = np.vstack([np.zeros((1, Z.shape[1])), np.cumsum(Z, axis=0)])   # fast window means
    t = np.arange(win, n - win)
    left = (csum[t] - csum[t - win]) / win
    right = (csum[t + win] - csum[t]) / win
    diff2 = (right - left) ** 2
    group_scores = np.zeros((n, len(FEATURE_GROUPS)), dtype=np.float32)
    for gi, cols in enumerate(FEATURE_GROUPS.values()):
        group_scores[t, gi] = np.sqrt(diff2[:, cols].mean(axis=1))
    return group_scores.max(axis=1), group_scores

def detect_splices(y, sr=SR, window_seconds=0.5, threshold_factor=3.0,
                   min_separation_seconds=0.75):
    # Returns: times, scores, threshold, detections (DataFrame).
    # A detection is a candidate boundary, NOT proof of manipulation.
    feats, times = extract_features(y, sr)
    frames_per_sec = sr / HOP_LENGTH
    win = max(3, int(window_seconds * frames_per_sec))
    if len(feats) < 2 * win + 5:
        raise ValueError("Audio too short for the selected window size.")

    Z = robust_standardize(feats)
    scores, group_scores = discontinuity_scores(Z, win)

    valid = scores[win:len(scores) - win]
    med = np.median(valid)
    mad = np.median(np.abs(valid - med)) * 1.4826
    threshold = float(med + threshold_factor * mad)

    peaks, props = find_peaks(scores, height=threshold,
                              distance=max(1, int(min_separation_seconds * frames_per_sec)))
    det = pd.DataFrame({
        "timestamp_seconds": np.round(times[peaks], 3),
        "anomaly_score": np.round(props["peak_heights"], 3),
        "score_over_threshold": np.round(props["peak_heights"] / threshold, 2),
        "main_cue": [list(FEATURE_GROUPS)[i] for i in group_scores[peaks].argmax(axis=1)],
    }).sort_values("timestamp_seconds").reset_index(drop=True)
    return times, scores, threshold, det

## 4. Forensic visualization  *(Member 3)*

In [ ]:
def plot_forensic_analysis(y, sr, times, scores, threshold, detections=None,
                           ground_truth=None, title="Forensic analysis"):
    dur = len(y) / sr
    fig, ax = plt.subplots(3, 1, figsize=(15, 9), sharex=True)
    ax[0].plot(np.arange(len(y)) / sr, y, lw=0.4, color="#335")
    ax[0].set_ylabel("Amplitude"); ax[0].set_title(title)

    S_db = librosa.amplitude_to_db(np.abs(librosa.stft(y, n_fft=FRAME_LENGTH, hop_length=HOP_LENGTH)), ref=np.max)
    librosa.display.specshow(S_db, sr=sr, hop_length=HOP_LENGTH, x_axis="time", y_axis="hz", ax=ax[1], cmap="magma")
    ax[1].set_ylabel("Frequency (Hz)")

    ax[2].plot(times, scores, lw=1, label="Discontinuity score D(t)")
    ax[2].axhline(threshold, ls="--", c="k", label=f"Adaptive threshold ({threshold:.2f})")
    ax[2].set_ylabel("Anomaly score"); ax[2].set_xlabel("Time (s)")

    if ground_truth is not None:
        for i, g in enumerate(ground_truth):
            for a in ax: a.axvline(g, c="limegreen", lw=3, alpha=0.5, label="True edit" if (a is ax[2] and i == 0) else None)
    if detections is not None:
        for i, t in enumerate(detections["timestamp_seconds"]):
            for a in ax: a.axvline(t, c="red", ls="--", lw=1.3, label="Flagged boundary" if (a is ax[2] and i == 0) else None)
    ax[2].legend(loc="upper right")
    for a in ax: a.set_xlim(0, dur)
    plt.tight_layout(); plt.show()

## 5. Evaluation metrics  *(Member 2)*

A detection is a **true positive** if it lies within ±`tolerance` seconds of a true edit boundary. Matching is **one-to-one** (each true boundary can be matched by only one detection), so duplicate alerts count as false positives instead of inflating the score.

In [ ]:
def match_boundaries(detected, truth, tolerance=0.25):
    # Greedy one-to-one matching, closest pairs first. Returns tp, fp, fn, errors.
    detected, truth = list(map(float, detected)), list(map(float, truth))
    pairs = sorted((abs(d - g), i, j) for i, d in enumerate(detected)
                   for j, g in enumerate(truth) if abs(d - g) <= tolerance)
    used_d, used_g, errors = set(), set(), []
    for err, i, j in pairs:
        if i not in used_d and j not in used_g:
            used_d.add(i); used_g.add(j); errors.append(err)
    tp = len(errors)
    return tp, len(detected) - tp, len(truth) - tp, errors

def summarize(tp, fp, fn, errors):
    p = tp / (tp + fp) if tp + fp else 0.0
    r = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * p * r / (p + r) if p + r else 0.0
    return {"TP": tp, "FP": fp, "FN": fn, "Precision": round(p, 3), "Recall": round(r, 3),
            "F1": round(f1, 3),
            "Mean loc. error (s)": round(float(np.mean(errors)), 3) if errors else None}

## 6. Sanity check on a simple synthetic signal

A tone-like section A (0–4 s), a different section B inserted (4–7 s), then A again (7–12 s). True boundaries: **4 s and 7 s**. This only checks that the code works — real evaluation is in section 8.

In [ ]:
def tone_section(duration, freqs, noise):
    t = np.arange(int(duration * SR)) / SR
    s = sum(np.sin(2 * np.pi * f * t) / (i + 1) for i, f in enumerate(freqs))
    s = s + noise * RNG.normal(size=len(t))
    return (0.5 * s / np.max(np.abs(s))).astype(np.float32)

sec_a = tone_section(9, [180, 360, 540], 0.015)
sec_b = tone_section(3, [500, 850, 1200], 0.05)
demo_audio = np.concatenate([sec_a[:4 * SR], sec_b, sec_a[4 * SR:]])
demo_truth = [4.0, 7.0]
display(Audio(demo_audio, rate=SR))

times, scores, thr, det = detect_splices(demo_audio)
display(det)
plot_forensic_analysis(demo_audio, SR, times, scores, thr, det, demo_truth, "Synthetic sanity check")
print(summarize(*match_boundaries(det.timestamp_seconds, demo_truth)))

## 7. Build a labelled speech dataset with known edit locations  *(Member 2)*

We use **real speech from LibriSpeech** (three clips from different speakers that ship with librosa — `libri1`, `libri2`, `libri3`). From them we generate:

| Condition | How it's made | What it tests |
|---|---|---|
| `genuine` | unedited excerpt | false alarms |
| `cross_speaker` | segment from a *different* recording inserted | typical forgery (different mic/room) |
| `same_recording` | segment from *the same* recording, other time | hardest case — same mic, same room |
| `noise_mismatch` | inserted segment has slightly different background noise (SNR 35 dB) | room/background inconsistency |
| `crossfade` | cross-speaker insert with 50 ms crossfades | forger hiding the cut |
| `gain_matched` | cross-speaker insert, loudness equalized | forger hiding level jump |

Every inserted segment creates **two** boundaries (start and end). The ground truth is saved to `ground_truth.csv`, and all audio to `dataset/`.

> If the librosa examples cannot be downloaded, the cell falls back to synthetic speech-like signals and tells you. **For your final report, use real recordings** — you can also put your own `.wav` files in `my_recordings/` and they are added automatically.

In [ ]:
import os, glob
os.makedirs("dataset", exist_ok=True)
os.makedirs("my_recordings", exist_ok=True)

def synthetic_speaker(seconds, f0, formant_shift, noise_db, cutoff, seed):
    # Fallback 'speech-like' generator (pulse train through moving formants + syllable envelope).
    rng = np.random.default_rng(seed)
    n = int(seconds * SR); y = np.zeros(n, np.float32)
    pos = 0
    while pos < n:
        syl = int(rng.uniform(0.12, 0.3) * SR)
        if rng.random() < 0.15:                      # pause
            pos += int(rng.uniform(0.1, 0.4) * SR); continue
        t = np.arange(syl) / SR
        f = f0 * rng.uniform(0.85, 1.2)
        src = np.sign(np.sin(2 * np.pi * f * t)) * 0.3 + 0.05 * rng.normal(size=syl)
        F1, F2 = rng.uniform(300, 800) * formant_shift, rng.uniform(900, 2300) * formant_shift
        spec = np.fft.rfft(src); fr = np.fft.rfftfreq(syl, 1 / SR)
        H = np.exp(-((fr - F1) / 120) ** 2) + 0.6 * np.exp(-((fr - F2) / 180) ** 2)
        s = np.fft.irfft(spec * H, syl) * np.hanning(syl)
        end = min(n, pos + syl); y[pos:end] += s[:end - pos]; pos = end
    y = y / (np.max(np.abs(y)) + 1e-8) * 0.6
    y_f = np.fft.irfft(np.fft.rfft(y) * (np.fft.rfftfreq(n, 1 / SR) < cutoff), n)   # channel bandwidth
    return (y_f + 10 ** (noise_db / 20) * rng.normal(size=n)).astype(np.float32)

def load_source_recordings():
    srcs = {}
    for key in ["libri1", "libri2", "libri3"]:
        try:
            y, _ = librosa.load(librosa.ex(key), sr=SR, mono=True)
            srcs[key] = y
        except Exception as e:
            print(f"Could not load {key}: {e.__class__.__name__}")
    for path in sorted(glob.glob("my_recordings/*.wav") + glob.glob("my_recordings/*.mp3")):
        y, _ = librosa.load(path, sr=SR, mono=True)
        if len(y) > 8 * SR: srcs[os.path.basename(path)] = y
    if len(srcs) < 2:
        print("WARNING: using SYNTHETIC speech-like signals. Results are NOT valid for your report.")
        srcs = {"synthA": synthetic_speaker(20, 120, 1.0, -55, 7000, 1),
                "synthB": synthetic_speaker(20, 210, 1.15, -48, 5500, 2),
                "synthC": synthetic_speaker(20, 150, 0.95, -60, 6500, 3)}
    for k, v in srcs.items(): print(f"{k:>20s}: {len(v)/SR:5.1f} s")
    return srcs

SOURCES = load_source_recordings()

In [ ]:
CLIP_SECONDS = 8.0

def excerpt(y, seconds, rng):
    n = int(seconds * SR); start = rng.integers(0, len(y) - n)
    return y[start:start + n].copy()

def insert_segment(host, seg, at, fade=0.0):
    # Insert seg into host at time `at`. Optional linear crossfade (seconds).
    a = int(at * SR); f = int(fade * SR)
    if f == 0:
        return np.concatenate([host[:a], seg, host[a:]])
    ramp = np.linspace(0, 1, f, dtype=np.float32)
    seg = seg.copy()
    left = host[:a].copy();  right = host[a:].copy()
    left[-f:] = left[-f:] * (1 - ramp) + seg[:f] * ramp           # fade into segment
    tail = seg[-f:] * (1 - ramp) + right[:f] * ramp                # fade back out
    return np.concatenate([left, seg[f:-f], tail, right[f:]])

def make_dataset(sources, per_condition=4, seed=7):
    rng = np.random.default_rng(seed)
    names = list(sources); rows = []
    conditions = ["genuine", "cross_speaker", "same_recording", "noise_mismatch", "crossfade", "gain_matched"]
    for host_name in names:
        host_src = sources[host_name]
        for cond in conditions:
            for k in range(per_condition):
                host = excerpt(host_src, CLIP_SECONDS, rng)
                truth = []
                if cond != "genuine":
                    seg_len = rng.uniform(1.5, 2.5)
                    if cond == "same_recording":
                        donor = host_src
                    else:
                        donor = sources[rng.choice([n for n in names if n != host_name])]
                    seg = excerpt(donor, seg_len, rng)
                    if cond == "noise_mismatch":
                        seg = excerpt(host_src, seg_len, rng)         # same speaker...
                        seg = seg + (np.sqrt(np.mean(seg**2)) / 10**(35/20)) * rng.normal(size=len(seg)).astype(np.float32)  # ...different background
                    if cond == "gain_matched":
                        seg = seg * (np.sqrt(np.mean(host**2)) / (np.sqrt(np.mean(seg**2)) + 1e-8))
                    at = rng.uniform(1.5, CLIP_SECONDS - 1.5)
                    fade = 0.05 if cond == "crossfade" else 0.0
                    y = insert_segment(host, seg.astype(np.float32), at, fade)
                    truth = [round(at, 3), round(at + len(seg) / SR - (fade if fade else 0), 3)]
                    if fade:   # crossfade shortens the clip by 2*fade; boundaries are the fade centres
                        truth = [round(at - fade / 2, 3), round(at + len(seg) / SR - 1.5 * fade, 3)]
                else:
                    y = host
                y = np.clip(y, -1, 1).astype(np.float32)
                fname = f"dataset/{cond}_{host_name.split('.')[0]}_{k}.wav"
                sf.write(fname, y, SR)
                rows.append({"file": fname, "condition": cond, "host": host_name,
                             "duration_s": round(len(y) / SR, 2),
                             "boundaries": ";".join(map(str, truth))})
    return pd.DataFrame(rows)

GT = make_dataset(SOURCES)
GT.to_csv("ground_truth.csv", index=False)
print(f"{len(GT)} clips created")
display(GT.groupby("condition").size().rename("clips").to_frame())
display(GT.head())

Listen to one spliced example and look at where the edit is:

In [ ]:
ex = GT[GT.condition == "cross_speaker"].iloc[0]
y_ex, _ = librosa.load(ex.file, sr=SR)
truth_ex = [float(b) for b in ex.boundaries.split(";")]
print(ex.file, "| true boundaries:", truth_ex)
display(Audio(y_ex, rate=SR))
t, s, thr, det = detect_splices(y_ex)
display(det)
plot_forensic_analysis(y_ex, SR, t, s, thr, det, truth_ex, f"{ex.condition}: {ex.file}")

## 8. Performance evaluation  *(this is what earns the marks)*

**Honest protocol:**
1. **Split by source recording**, not by clip: clips made from the first source are the *validation* set, the rest are the *test* set. This prevents near-identical audio leaking between tune and test.
2. Choose `threshold_factor` on validation only (best F1).
3. Report test results per condition, plus false alarms per minute on genuine audio and processing time.

In [ ]:
TOLERANCE = 0.25   # seconds - fixed BEFORE looking at results

def run_eval(df, threshold_factor, window_seconds=0.5):
    rows = []
    for r in df.itertuples():
        y, _ = librosa.load(r.file, sr=SR)
        truth = [float(b) for b in r.boundaries.split(";")] if r.boundaries else []
        t0 = time.perf_counter()
        _, _, _, det = detect_splices(y, threshold_factor=threshold_factor, window_seconds=window_seconds)
        elapsed = time.perf_counter() - t0
        tp, fp, fn, err = match_boundaries(det.timestamp_seconds, truth, TOLERANCE)
        rows.append({"file": r.file, "condition": r.condition, "duration_s": r.duration_s,
                     "tp": tp, "fp": fp, "fn": fn, "errors": err, "time_s": elapsed,
                     "detected": det.timestamp_seconds.tolist(), "truth": truth})
    return pd.DataFrame(rows)

def aggregate(res):
    out = summarize(res.tp.sum(), res.fp.sum(), res.fn.sum(), sum(res.errors, []))
    out["FA / min"] = round(res.fp.sum() / (res.duration_s.sum() / 60), 2)
    out["Clips"] = len(res)
    return out

hosts = list(GT.host.unique())
VAL = GT[GT.host == hosts[0]]
TEST = GT[GT.host != hosts[0]]
print(f"Validation: {len(VAL)} clips from {hosts[0]} | Test: {len(TEST)} clips from {hosts[1:]}")

sweep = []
for k in [2.0, 2.5, 3.0, 3.5, 4.0, 5.0, 6.0]:
    agg = aggregate(run_eval(VAL, k)); agg["threshold_factor"] = k; sweep.append(agg)
sweep = pd.DataFrame(sweep).set_index("threshold_factor")
display(sweep)
BEST_K = float(sweep["F1"].idxmax())
print("Chosen threshold_factor (validation F1):", BEST_K)

In [ ]:
# Precision/recall trade-off on the validation set
plt.figure(figsize=(7, 4))
plt.plot(sweep.index, sweep.Precision, "o-", label="Precision")
plt.plot(sweep.index, sweep.Recall, "s-", label="Recall")
plt.plot(sweep.index, sweep.F1, "^-", label="F1")
plt.axvline(BEST_K, ls="--", c="gray")
plt.xlabel("threshold_factor k"); plt.ylabel("score"); plt.title("Validation sweep")
plt.legend(); plt.grid(alpha=.3); plt.tight_layout(); plt.show()

In [ ]:
# ---------------- FINAL TEST RESULTS ----------------
test_res = run_eval(TEST, BEST_K)
per_cond = pd.DataFrame({c: aggregate(g) for c, g in test_res.groupby("condition")}).T
per_cond.loc["ALL (spliced only)"] = pd.Series(aggregate(test_res[test_res.condition != "genuine"]))
per_cond.loc["ALL"] = pd.Series(aggregate(test_res))
print(f"Test set, threshold_factor={BEST_K}, tolerance=±{TOLERANCE}s")
display(per_cond)
print(f"Mean processing time: {test_res.time_s.mean()*1000:.0f} ms per {CLIP_SECONDS:.0f}-s clip")
per_cond.to_csv("results_per_condition.csv")
test_res.to_csv("results_per_clip.csv", index=False)

In [ ]:
# Localization error histogram + bar chart of recall per condition
errs = sum(test_res.errors, [])
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].hist(np.array(errs) * 1000, bins=15, color="#4a7")
ax[0].set_xlabel("Localization error (ms)"); ax[0].set_ylabel("Count"); ax[0].set_title("Matched boundaries")
pc = per_cond.drop(index=["genuine", "ALL", "ALL (spliced only)"], errors="ignore")
ax[1].bar(pc.index, pc.Recall.astype(float), color="#47a")
ax[1].set_ylabel("Recall"); ax[1].set_ylim(0, 1); ax[1].set_title("Recall by splice type")
ax[1].tick_params(axis="x", rotation=30)
plt.tight_layout(); plt.show()

### Which features actually detect splices?  *(great slide for the viva)*
On the **validation** clips, compare how much each feature jumps at a *true* boundary vs. at ordinary positions. Ratio ≫ 1 means the feature is a useful splice cue.

In [ ]:
def feature_separability(df, win_s=0.5):
    win = int(win_s * SR / HOP_LENGTH); at, rest = [], []
    for r in df[df.boundaries != ""].itertuples():
        y, _ = librosa.load(r.file, sr=SR)
        Z = robust_standardize(extract_features(y)[0]); n = len(Z)
        truth = [int(float(b) * SR / HOP_LENGTH) for b in r.boundaries.split(";")]
        jump = lambda i: np.abs(Z[i:i + win].mean(0) - Z[i - win:i].mean(0))
        at += [jump(b) for b in truth if win < b < n - win]
        rest += [jump(i) for i in range(win, n - win, 10) if min(abs(i - b) for b in truth) > 1.2 * win]
    at, rest = np.mean(at, 0), np.mean(rest, 0)
    return pd.DataFrame({"jump_at_splice": at, "jump_elsewhere": rest, "ratio": at / rest},
                        index=FEATURE_NAMES).round(2).sort_values("ratio", ascending=False)

sep = feature_separability(VAL)
display(sep)
sep["ratio"].plot.barh(figsize=(7, 6), color="#a64", title="Splice-boundary separability (validation)")
plt.axvline(1, c="k", ls="--"); plt.gca().invert_yaxis(); plt.tight_layout(); plt.show()

### Failure analysis
Show your examiners where the system fails — it's part of a good evaluation. The cell below plots the worst missed splice and the worst false alarm from the test set.

In [ ]:
def show_case(row, label):
    y, _ = librosa.load(row.file, sr=SR)
    t, s, thr, det = detect_splices(y, threshold_factor=BEST_K)
    plot_forensic_analysis(y, SR, t, s, thr, det, row.truth, f"{label}: {row.file}")
    display(Audio(y, rate=SR))

missed = test_res[test_res.fn > 0].sort_values("fn", ascending=False)
false_alarm = test_res[test_res.fp > 0].sort_values("fp", ascending=False)
if len(missed): show_case(missed.iloc[0], "MISSED SPLICE")
if len(false_alarm): show_case(false_alarm.iloc[0], "FALSE ALARM")

## 9. Analyze your own recording + export report  *(demo)*

Upload any WAV/MP3. The cell prints the suspicious boundaries, plots the evidence, lets you listen to ±1 s around each boundary, and saves a CSV report.

In [ ]:
def analyze_file(path, threshold_factor=None):
    k = threshold_factor or BEST_K
    y, _ = librosa.load(path, sr=SR, mono=True)
    t, s, thr, det = detect_splices(y, threshold_factor=k)
    print(f"File: {path} | duration {len(y)/SR:.2f} s | threshold_factor {k} | "
          f"{len(det)} suspicious boundary(ies)")
    display(det)
    plot_forensic_analysis(y, SR, t, s, thr, det, title=f"Analysis of {os.path.basename(path)}")
    for i, ts in enumerate(det.timestamp_seconds):
        a, b = max(0, int((ts - 1) * SR)), min(len(y), int((ts + 1) * SR))
        print(f"Boundary {i+1} at {ts:.2f} s (listen to {max(0, ts-1):.1f}-{min(len(y)/SR, ts+1):.1f} s):")
        display(Audio(y[a:b], rate=SR))
    report = det.copy()
    report.insert(0, "file", os.path.basename(path))
    report["threshold_factor"] = k
    report["note"] = "Candidate boundary from acoustic discontinuity; not proof of tampering."
    out = f"report_{os.path.splitext(os.path.basename(path))[0]}.csv"
    report.to_csv(out, index=False); print("Report saved:", out)
    return report

AUDIO_EXT = (".wav", ".mp3", ".flac", ".ogg", ".m4a")
if IN_COLAB:
    print("Choose an AUDIO file (WAV/MP3/FLAC/OGG/M4A) - not the notebook.")
    up = files.upload()
    for name in up:
        if name.lower().endswith(AUDIO_EXT):
            analyze_file(name)
        else:
            print(f"Skipped '{name}': not an audio file. Upload a recording such as a .wav or .mp3.")
else:
    analyze_file(GT[GT.condition == "cross_speaker"].iloc[-1].file)

## 10. AudioGuard web app — for the live presentation

Run this cell: it prints a public **gradio.live** link. Open it in a browser tab before your demo.

- **Left:** upload or record audio, set the threshold, or click an example from your test set.
- **Right:** verdict, key numbers, and four tabs — *Evidence* (interactive waveform / spectrogram / score; hover for values, drag to zoom), *Boundaries* (each flag with its strength and main cue), *Listen* (±1 s around any flag), *Report* (CSV download).

In [ ]:
import inspect
import gradio as gr
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ---------------- Look & feel (matches the presentation deck) ----------------
AG = dict(bg="#0E1A27", panel="#14202E", raised="#1A2A3B", line="#26384C", text="#E8EEF3",
          muted="#8FA3B5", teal="#13A89E", orange="#E4572E", amber="#F2B134", blue="#5B7DB1")
CUE_COLORS = {"MFCC (timbre / channel)": AG["blue"], "Level": AG["amber"],
              "Spectral shape": "#A99BF0", "Background noise": AG["orange"]}

_theme_vars = dict(
    body_background_fill=AG["bg"], body_text_color=AG["text"], body_text_color_subdued=AG["muted"],
    background_fill_primary=AG["panel"], background_fill_secondary=AG["raised"],
    block_background_fill=AG["panel"], block_border_color=AG["line"], border_color_primary=AG["line"],
    block_label_background_fill=AG["raised"], block_label_text_color=AG["muted"],
    block_title_text_color=AG["text"], input_background_fill=AG["raised"], input_border_color=AG["line"],
    button_primary_background_fill=AG["teal"], button_primary_background_fill_hover="#0F8F86",
    button_primary_text_color="#FFFFFF", button_secondary_background_fill=AG["raised"],
    button_secondary_text_color=AG["text"], slider_color=AG["teal"], color_accent=AG["teal"],
    table_even_background_fill=AG["panel"], table_odd_background_fill=AG["raised"],
    table_border_color=AG["line"],
)
# Same colours in light and dark mode; keep only the variables this Gradio version knows.
_known = set(inspect.signature(gr.themes.Base.set).parameters)
AG_THEME = gr.themes.Base(
    primary_hue="teal", neutral_hue="slate",
    font=[gr.themes.GoogleFont("IBM Plex Sans"), "system-ui", "sans-serif"],
    font_mono=[gr.themes.GoogleFont("IBM Plex Mono"), "ui-monospace", "monospace"],
).set(**{k + sfx: v for k, v in _theme_vars.items() for sfx in ("", "_dark") if k + sfx in _known})

AG_CSS = """
.gradio-container {max-width: 1280px !important; margin: 0 auto !important;}
footer {display: none !important;}
.ag-header svg {flex:none;}
.ag-header {display:flex; align-items:center; gap:16px; padding:8px 4px 4px;}
.ag-header h1 {margin:0; font-size:28px; font-weight:700; letter-spacing:-0.01em; color:#E8EEF3;}
.ag-header p {margin:2px 0 0; color:#8FA3B5; font-size:15px;}
.ag-badge {margin-left:auto; border:1px solid #26384C; color:#F2B134; border-radius:999px;
           padding:4px 12px; font-size:12px; text-transform:uppercase; letter-spacing:.08em;}
.ag-note {color:#8FA3B5; font-size:13px; line-height:1.5; padding:4px 2px;}
.ag-empty {border:1px dashed #26384C; border-radius:14px; padding:40px 24px; text-align:center; color:#8FA3B5;}
.ag-empty b {color:#E8EEF3; display:block; font-size:17px; margin-bottom:6px;}
.ag-verdict {display:flex; align-items:center; gap:12px; border-radius:14px; padding:14px 18px;
             font-size:16px; font-weight:600; margin-bottom:12px;}
.ag-verdict.alert {background:rgba(228,87,46,.12); border:1px solid rgba(228,87,46,.5); color:#FF9B7A;}
.ag-verdict.clear {background:rgba(19,168,158,.12); border:1px solid rgba(19,168,158,.5); color:#5FD6CC;}
.ag-verdict .dot {width:10px; height:10px; border-radius:50%; background:currentColor; flex:none;}
.ag-verdict .file {margin-left:auto; font-weight:400; font-size:13px; color:#8FA3B5;
                   font-family:'IBM Plex Mono',monospace; overflow:hidden; text-overflow:ellipsis; white-space:nowrap; max-width:45%;}
.ag-tiles {display:grid; grid-template-columns:repeat(auto-fit,minmax(150px,1fr)); gap:12px;}
.ag-tile {background:#1A2A3B; border:1px solid #26384C; border-radius:14px; padding:14px 16px;}
.ag-tile .v {font-family:'IBM Plex Mono',monospace; font-size:26px; font-weight:600; color:#E8EEF3; line-height:1.2;}
.ag-tile .v.txt {font-family:'IBM Plex Sans',sans-serif; font-size:20px; padding-top:4px;}
.ag-tile .l {font-size:12px; color:#8FA3B5; text-transform:uppercase; letter-spacing:.06em; margin-top:4px;}
.ag-cards {display:flex; flex-direction:column; gap:10px;}
.ag-card {display:grid; grid-template-columns:110px 1fr auto; align-items:center; gap:16px;
          background:#1A2A3B; border:1px solid #26384C; border-radius:12px; padding:12px 16px;}
.ag-card .ts {font-family:'IBM Plex Mono',monospace; font-size:20px; font-weight:600; color:#E8EEF3;}
.ag-card .ts small {display:block; font-size:11px; color:#8FA3B5; font-weight:400; font-family:'IBM Plex Sans',sans-serif;}
.ag-bar {height:8px; background:#26384C; border-radius:999px; overflow:hidden;}
.ag-bar span {display:block; height:100%; border-radius:999px;}
.ag-card .meta {font-size:12px; color:#8FA3B5; margin-top:6px;}
.ag-chip {font-size:12px; padding:4px 10px; border-radius:999px; border:1px solid; white-space:nowrap;}
@media (max-width: 640px) {
  .ag-card {grid-template-columns:1fr; gap:8px;}
  .ag-verdict {flex-wrap:wrap;} .ag-verdict .file {margin-left:0; max-width:100%;}
  .ag-badge {display:none;}
}
"""

AG_LOGO = ('<svg width="44" height="44" viewBox="0 0 44 44" aria-hidden="true">'
           + "".join(f'<rect x="{3 + i * 5}" y="{22 - h / 2}" width="3" height="{h}" rx="1.5" fill="{AG["orange"] if 3 <= i <= 5 else AG["teal"]}"/>'
                     for i, h in enumerate([10, 20, 30, 16, 36, 24, 14, 22]))
           + "</svg>")
AG_HEADER = f"""<div class="ag-header">{AG_LOGO}<div><h1>AudioGuard</h1>
<p>Audio splicing forensic analyzer — finds where a recording may have been edited, and why</p></div>
<span class="ag-badge">Prototype</span></div>"""
AG_EMPTY = """<div class="ag-empty"><b>No recording analyzed yet</b>
Upload a WAV/MP3, or pick one of the examples, and press <i>Analyze recording</i>.</div>"""


def evidence_figure(y, sr, t, s, thr, det):
    fig = make_subplots(rows=3, cols=1, shared_xaxes=True, vertical_spacing=0.07,
                        row_heights=[0.26, 0.42, 0.32],
                        subplot_titles=("Waveform", "Spectrogram", "Discontinuity score D(t)"))
    # waveform as a min/max envelope (fast even for long files)
    hop = max(1, len(y) // 2000)
    yy = y[:hop * (len(y) // hop)].reshape(-1, hop)
    tx = (np.arange(len(yy)) * hop + hop / 2) / sr
    fig.add_trace(go.Scatter(x=np.r_[tx, tx[::-1]], y=np.r_[yy.max(1), yy.min(1)[::-1]], fill="toself",
                             fillcolor=AG["teal"], line=dict(width=0), hoverinfo="skip"), 1, 1)
    # spectrogram (down-sampled in time for the browser)
    S_db = librosa.amplitude_to_db(np.abs(librosa.stft(y, n_fft=FRAME_LENGTH, hop_length=HOP_LENGTH)), ref=np.max)
    step = max(1, S_db.shape[1] // 900)
    fig.add_trace(go.Heatmap(
        x=librosa.frames_to_time(np.arange(0, S_db.shape[1], step), sr=sr, hop_length=HOP_LENGTH),
        y=librosa.fft_frequencies(sr=sr, n_fft=FRAME_LENGTH), z=S_db[:, ::step],
        colorscale="Magma", zmin=-80, zmax=0, showscale=False,
        hovertemplate="%{x:.2f} s · %{y:.0f} Hz · %{z:.0f} dB<extra></extra>"), 2, 1)
    # score + threshold
    fig.add_trace(go.Scatter(x=t, y=s, line=dict(color=AG["text"], width=1.3),
                             hovertemplate="%{x:.2f} s · score %{y:.2f}<extra></extra>"), 3, 1)
    fig.add_hline(y=thr, line=dict(color=AG["amber"], dash="dash", width=1), row=3, col=1,
                  annotation_text=f"threshold {thr:.2f}", annotation_font_color=AG["amber"],
                  annotation_position="top left")
    for i, ts in enumerate(det.timestamp_seconds, 1):
        for r in (1, 2, 3):
            fig.add_vline(x=ts, line=dict(color=AG["orange"], width=1.6), row=r, col=1)
        fig.add_annotation(x=ts, y=1.0, yref="paper", text=f"#{i}", showarrow=False, yshift=12,
                           font=dict(color=AG["orange"], size=12))
    fig.update_layout(height=600, margin=dict(l=55, r=20, t=40, b=45), showlegend=False,
                      paper_bgcolor=AG["panel"], plot_bgcolor=AG["panel"], hovermode="closest",
                      font=dict(family="IBM Plex Sans, sans-serif", color=AG["muted"], size=12))
    fig.update_annotations(font_color=AG["text"], selector=dict(text="Waveform"))
    fig.update_xaxes(gridcolor=AG["line"], zeroline=False, range=[0, len(y) / sr])
    fig.update_yaxes(gridcolor=AG["line"], zeroline=False)
    fig.update_yaxes(title_text="Hz", row=2, col=1)
    fig.update_xaxes(title_text="Time (s)", row=3, col=1)
    return fig


def summary_html(name, dur, det, elapsed):
    n = len(det)
    if n:
        verdict = f'<div class="ag-verdict alert"><span class="dot"></span>{n} suspicious boundar{"y" if n == 1 else "ies"} — review the evidence<span class="file">{name}</span></div>'
        cue = det.main_cue.value_counts().idxmax()
    else:
        verdict = f'<div class="ag-verdict clear"><span class="dot"></span>No suspicious boundaries above threshold<span class="file">{name}</span></div>'
        cue = "—"
    cue_style = f' style="color:{CUE_COLORS.get(cue, AG["text"])}"'
    tiles = [(f"{dur:.1f} s", "Duration", "", ""), (str(n), "Flagged boundaries", "", ""),
             (cue.split(" (")[0], "Main cue", " txt", cue_style), (f"{elapsed * 1000:.0f} ms", "Analysis time", "", "")]
    return verdict + '<div class="ag-tiles">' + "".join(
        f'<div class="ag-tile"><div class="v{c}"{st}>{v}</div><div class="l">{l}</div></div>'
        for v, l, c, st in tiles) + "</div>"


def cards_html(det):
    if not len(det):
        return '<div class="ag-empty"><b>Nothing flagged</b>No point in this recording crossed the adaptive threshold.</div>'
    top = det.anomaly_score.max()
    out = []
    for i, r in enumerate(det.itertuples(), 1):
        col = CUE_COLORS.get(r.main_cue, AG["muted"])
        out.append(f"""<div class="ag-card"><div class="ts">{r.timestamp_seconds:.2f} s<small>boundary #{i}</small></div>
<div><div class="ag-bar"><span style="width:{100 * r.anomaly_score / top:.0f}%;background:{col}"></span></div>
<div class="meta">score {r.anomaly_score:.2f} · {r.score_over_threshold:.2f}× threshold</div></div>
<span class="ag-chip" style="color:{col};border-color:{col}">{r.main_cue}</span></div>""")
    return '<div class="ag-cards">' + "".join(out) + "</div>"


def ag_analyze(path, k):
    if not path:
        raise gr.Error("Upload a recording first.")
    t0 = time.perf_counter()
    try:
        y, _ = librosa.load(path, sr=SR, mono=True)
    except Exception:
        raise gr.Error("Could not read that file as audio. Please upload a WAV, MP3, FLAC or OGG recording.")
    if len(y) < 3 * SR:
        raise gr.Error("Recording is too short — please use at least 3 seconds of audio.")
    t, s, thr, det = detect_splices(y, threshold_factor=k)
    elapsed = time.perf_counter() - t0
    name = os.path.basename(path)

    report = det.copy()
    report.insert(0, "file", name)
    report["threshold_factor"] = k
    report["note"] = "Candidate boundary from acoustic discontinuity; not proof of tampering."
    report_path = f"audioguard_report_{os.path.splitext(name)[0]}.csv"
    report.to_csv(report_path, index=False)

    choices = [f"#{i} · {ts:.2f} s · {cue}" for i, (ts, cue) in
               enumerate(zip(det.timestamp_seconds, det.main_cue), 1)]
    first = choices[0] if choices else None
    return (summary_html(name, len(y) / SR, det, elapsed), evidence_figure(y, SR, t, s, thr, det),
            cards_html(det), det, gr.Dropdown(choices=choices, value=first),
            ag_snippet(first, (y, det)), report_path, (y, det))


def ag_snippet(choice, state):
    if not choice or state is None:
        return None
    y, det = state
    ts = float(det.timestamp_seconds.iloc[int(choice.split(" ")[0][1:]) - 1])
    a, b = max(0, int((ts - 1) * SR)), min(len(y), int((ts + 1) * SR))
    return (SR, y[a:b])


_style = dict(theme=AG_THEME, css=AG_CSS)
_blocks_takes_style = "css" in inspect.signature(gr.Blocks.__init__).parameters   # Gradio 4/5 vs 6
with gr.Blocks(title="AudioGuard", **(_style if _blocks_takes_style else {})) as demo:
    state = gr.State()
    gr.HTML(AG_HEADER)
    with gr.Row(equal_height=False):
        with gr.Column(scale=4, min_width=300):
            audio_in = gr.Audio(type="filepath", sources=["upload", "microphone"], label="Recording")
            k_in = gr.Slider(1.5, 6.0, value=BEST_K, step=0.5, label="Detection threshold k",
                             info="Lower = more sensitive, but more false alarms")
            run_btn = gr.Button("Analyze recording", variant="primary", size="lg")
            gr.Examples(label="Examples from our test set",
                        examples=[[GT[GT.condition == "cross_speaker"].iloc[0].file],
                                  [GT[GT.condition == "gain_matched"].iloc[0].file],
                                  [GT[GT.condition == "genuine"].iloc[0].file]],
                        example_labels=["Spliced · different recording", "Spliced · loudness matched", "Genuine"],
                        inputs=[audio_in])
            gr.HTML('<div class="ag-note">A flagged boundary is a place where the acoustic properties '
                    'change abruptly. It is evidence to examine, not proof of tampering.</div>')
        with gr.Column(scale=8, min_width=320):
            summary_out = gr.HTML(AG_EMPTY)
            with gr.Tabs():
                with gr.Tab("Evidence"):
                    plot_out = gr.Plot(show_label=False)
                with gr.Tab("Boundaries"):
                    cards_out = gr.HTML()
                    table_out = gr.Dataframe(label="Raw values", interactive=False)
                with gr.Tab("Listen"):
                    pick = gr.Dropdown(label="Flagged boundary", choices=[], interactive=True)
                    snippet_out = gr.Audio(label="±1 second around the boundary", interactive=False)
                with gr.Tab("Report"):
                    report_out = gr.File(label="CSV report")

    outputs = [summary_out, plot_out, cards_out, table_out, pick, snippet_out, report_out, state]
    run_btn.click(ag_analyze, [audio_in, k_in], outputs)
    pick.change(ag_snippet, [pick, state], snippet_out)

if IN_COLAB:
    demo.launch(share=True, debug=False, **({} if _blocks_takes_style else _style))

## 11. Limitations and future work (for your slides)
- Feature means/medians need ~0.5 s of audio on each side, so edits shorter than ~1 s or very close to the file edges are hard to catch.
- Splices **within the same recording** (same mic, same room) leave few acoustic traces — expect low recall there; this is a known open problem.
- Natural events (speaker change in a genuine conversation, door slam, music) can trigger false alarms.
- Lossy re-compression (MP3/WhatsApp) can mask or create discontinuities — test it as an extra condition if you have time.
- **Future work:** ENF (electrical network frequency) phase analysis, reverberation/RT60 consistency, double-compression detection, and a learned model (e.g. CNN on log-mel patches) trained on a larger corpus.